# EduRAG Sprint 3 Batch 5 - One-PDF RAG PoC

This notebook runs the first end-to-end EduRAG proof of concept on one real PDF: `materials/book_embNLP_draft.pdf`.

Minimum success criteria covered here: ingest, chunk, embed, store with `course_id`, retrieve top-K, generate a grounded answer, show source citations, and show fallback behaviour when retrieval confidence is low.

## 1. Configuration and Dependency Check

In [1]:
from pathlib import Path
import importlib.util
import json
import os
import re
import socket
import textwrap
import urllib.error
import urllib.request

required_modules = {
    "pypdf": "pypdf",
    "sentence_transformers": "sentence-transformers",
    "langchain_text_splitters": "langchain-text-splitters",
    "chromadb": "chromadb",
    "requests": "requests",
    "numpy": "numpy",
}
missing = [pip_name for module_name, pip_name in required_modules.items() if importlib.util.find_spec(module_name) is None]
if missing:
    raise RuntimeError(
        "Missing required packages. Install them with: python -m pip install " + " ".join(sorted(set(missing)))
    )

import chromadb
import numpy as np
import requests
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PDF_PATH = PROJECT_ROOT / "materials" / "book_embNLP_draft.pdf"
COURSE_ID = os.getenv("EDURAG_COURSE_ID", "CSIT321")
DOCUMENT_NAME = PDF_PATH.name

POSTGRES_HOST = os.getenv("POSTGRES_HOST", "localhost")
POSTGRES_PORT = int(os.getenv("POSTGRES_PORT", "5432"))
CHROMA_HOST = os.getenv("CHROMA_HOST", "localhost")
CHROMA_PORT = int(os.getenv("CHROMA_PORT", "8000"))
CHROMA_BASE_URL = os.getenv("CHROMA_BASE_URL", f"http://{CHROMA_HOST}:{CHROMA_PORT}")
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
DEFAULT_OLLAMA_MODEL = os.getenv("DEFAULT_OLLAMA_MODEL", "qwen3:14b")

COLLECTION_NAME = f"edurag_{COURSE_ID.lower()}_sprint3_poc"
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
CHUNK_SIZE = 1024
CHUNK_OVERLAP = 0
TOP_K = 5
RELEVANCE_DISTANCE_THRESHOLD = 0.65
FALLBACK_LABEL = "Based on general knowledge - not verified against course materials"

config = {
    "project_root": ".",
    "pdf_path": str(PDF_PATH.relative_to(PROJECT_ROOT)),
    "pdf_exists": PDF_PATH.exists(),
    "course_id": COURSE_ID,
    "collection_name": COLLECTION_NAME,
    "embedding_model": EMBEDDING_MODEL_NAME,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "top_k": TOP_K,
    "relevance_distance_threshold": RELEVANCE_DISTANCE_THRESHOLD,
    "chunker": "LangChain RecursiveCharacterTextSplitter",
    "chromadb": CHROMA_BASE_URL,
    "ollama": {"base_url": OLLAMA_BASE_URL, "preferred_model": DEFAULT_OLLAMA_MODEL},
}

print(json.dumps(config, indent=2))



{
  "project_root": ".",
  "pdf_path": "materials/book_embNLP_draft.pdf",
  "pdf_exists": true,
  "course_id": "CSIT321",
  "collection_name": "edurag_csit321_sprint3_poc",
  "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
  "chunk_size": 1024,
  "chunk_overlap": 0,
  "top_k": 5,
  "relevance_distance_threshold": 0.65,
  "chunker": "LangChain RecursiveCharacterTextSplitter",
  "chromadb": "http://localhost:8000",
  "ollama": {
    "base_url": "http://localhost:11434",
    "preferred_model": "qwen3:14b"
  }
}


## 2. Service Connectivity Checks

In [2]:
def check_tcp(host: str, port: int, timeout: float = 2.0) -> dict:
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return {"ok": True, "detail": "TCP port reachable"}
    except OSError as exc:
        return {"ok": False, "detail": str(exc)}


def check_http_json(url: str, timeout: float = 4.0) -> dict:
    try:
        with urllib.request.urlopen(url, timeout=timeout) as response:
            raw = response.read().decode("utf-8", errors="replace")
            try:
                body = json.loads(raw)
            except json.JSONDecodeError:
                body = raw[:300]
            return {"ok": True, "status": response.status, "body": body}
    except (urllib.error.URLError, TimeoutError, OSError) as exc:
        return {"ok": False, "detail": str(exc)}


service_checks = {
    "postgres_tcp": check_tcp(POSTGRES_HOST, POSTGRES_PORT),
    "chromadb_tcp": check_tcp(CHROMA_HOST, CHROMA_PORT),
    "chromadb_v2_heartbeat": check_http_json(f"{CHROMA_BASE_URL}/api/v2/heartbeat"),
    "ollama_tags": check_http_json(f"{OLLAMA_BASE_URL}/api/tags"),
}

print(json.dumps(service_checks, indent=2))

{
  "postgres_tcp": {
    "ok": true,
    "detail": "TCP port reachable"
  },
  "chromadb_tcp": {
    "ok": true,
    "detail": "TCP port reachable"
  },
  "chromadb_v2_heartbeat": {
    "ok": true,
    "status": 200,
    "body": {
      "nanosecond heartbeat": 1776782364409069596
    }
  },
  "ollama_tags": {
    "ok": true,
    "status": 200,
    "body": {
      "models": [
        {
          "name": "qwen3:14b",
          "model": "qwen3:14b",
          "modified_at": "2026-04-21T12:44:35.297985416Z",
          "size": 9276198565,
          "digest": "bdbd181c33f2ed1b31c972991882db3cf4d192569092138a7d29e973cd9debe8",
          "details": {
            "parent_model": "",
            "format": "gguf",
            "family": "qwen3",
            "families": [
              "qwen3"
            ],
            "parameter_size": "14.8B",
            "quantization_level": "Q4_K_M"
          }
        }
      ]
    }
  }
}


## 3. Ingest PDF

In [3]:
if not PDF_PATH.exists():
    raise FileNotFoundError(f"PoC PDF not found: {PDF_PATH}")

reader = PdfReader(str(PDF_PATH))
raw_pages = []
for page_number, page in enumerate(reader.pages, start=1):
    text = " ".join((page.extract_text() or "").split())
    if text:
        raw_pages.append({"page_number": page_number, "text": text})

print(f"PDF pages in file: {len(reader.pages)}")
print(f"Pages with extracted text: {len(raw_pages)}")
print("First page preview:")
print(textwrap.shorten(raw_pages[0]["text"], width=300, placeholder="..."))

PDF pages in file: 163
Pages with extracted text: 163
First page preview:
DRAFT Embeddings in Natural Language Processing Theory and Advances in V ector Representation of Meaning Mohammad T aher Pilehvar T ehran Institute for Advanced Studies Jose Camacho-Collados Cardiff University SYNTHESIS LECTURES ON HUMAN LANGUAGE TECHNOLOGIES CM& cLaypoo lMorgan publishers&


## 4. Chunk Text with Course Metadata

In [4]:
def chunk_pages(pages, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
        keep_separator=True,
    )

    chunks = []
    for page in pages:
        page_chunks = [chunk.strip() for chunk in splitter.split_text(page["text"]) if chunk.strip()]
        for local_index, chunk_text in enumerate(page_chunks):
            if len(chunk_text) < 30 and local_index > 0:
                continue
            chunk_index = len(chunks)
            chunks.append({
                "id": f"{COURSE_ID}_{DOCUMENT_NAME}_p{page['page_number']:03d}_c{chunk_index:04d}",
                "text": chunk_text,
                "metadata": {
                    "course_id": COURSE_ID,
                    "document_name": DOCUMENT_NAME,
                    "page_number": page["page_number"],
                    "chunk_index": chunk_index,
                    "page_chunk_index": local_index,
                    "chunker": "RecursiveCharacterTextSplitter",
                    "chunk_size": chunk_size,
                    "chunk_overlap": overlap,
                },
            })
    return chunks


chunks = chunk_pages(raw_pages)
print(f"Chunk count: {len(chunks)}")
print("Required metadata on first chunk:")
print(json.dumps(chunks[0]["metadata"], indent=2))
print("First chunk preview:")
print(textwrap.shorten(chunks[0]["text"], width=300, placeholder="..."))



Chunk count: 464
Required metadata on first chunk:
{
  "course_id": "CSIT321",
  "document_name": "book_embNLP_draft.pdf",
  "page_number": 1,
  "chunk_index": 0,
  "page_chunk_index": 0,
  "chunker": "RecursiveCharacterTextSplitter",
  "chunk_size": 1024,
  "chunk_overlap": 0
}
First chunk preview:
DRAFT Embeddings in Natural Language Processing Theory and Advances in V ector Representation of Meaning Mohammad T aher Pilehvar T ehran Institute for Advanced Studies Jose Camacho-Collados Cardiff University SYNTHESIS LECTURES ON HUMAN LANGUAGE TECHNOLOGIES CM& cLaypoo lMorgan publishers&


## 5. Embed Chunks

In [5]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
chunk_texts = [chunk["text"] for chunk in chunks]
chunk_embeddings = embedding_model.encode(
    chunk_texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
)

print(f"Embedding matrix shape: {chunk_embeddings.shape}")
print(f"Embedding dtype: {chunk_embeddings.dtype}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/15 [00:00<?, ?it/s]

Embedding matrix shape: (464, 384)
Embedding dtype: float32


## 6. Store Chunks in ChromaDB

In [6]:
chroma_client = chromadb.HttpClient(host=CHROMA_HOST, port=CHROMA_PORT)

# Keep the PoC deterministic by replacing the collection on each run.
try:
    chroma_client.delete_collection(COLLECTION_NAME)
except Exception:
    pass

collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"purpose": "EduRAG Sprint 3 PoC with Batch 7 RAG defaults"},
)

collection.add(
    ids=[chunk["id"] for chunk in chunks],
    documents=chunk_texts,
    metadatas=[chunk["metadata"] for chunk in chunks],
    embeddings=chunk_embeddings.tolist(),
)

print(f"Stored chunks in ChromaDB collection: {COLLECTION_NAME}")
print(f"Collection count: {collection.count()}")

Stored chunks in ChromaDB collection: edurag_csit321_sprint3_poc
Collection count: 464


## 7. Retrieve Top-K Context

In [7]:
def retrieve(query: str, top_k: int = TOP_K):
    query_embedding = embedding_model.encode([query], normalize_embeddings=True).tolist()
    result = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k,
        where={"course_id": COURSE_ID},
        include=["documents", "metadatas", "distances"],
    )
    hits = []
    for document, metadata, distance in zip(result["documents"][0], result["metadatas"][0], result["distances"][0]):
        hits.append({
            "text": document,
            "metadata": metadata,
            "distance": float(distance),
        })
    return hits


sample_query = "What are word embeddings and why are they useful in NLP?"
retrieved_chunks = retrieve(sample_query)

print(f"Query: {sample_query}")
for index, hit in enumerate(retrieved_chunks, start=1):
    metadata = hit["metadata"]
    print(
        f"{index}. distance={hit['distance']:.4f} | "
        f"{metadata['document_name']} p.{metadata['page_number']} chunk {metadata['chunk_index']}"
    )
    print("   " + textwrap.shorten(hit["text"], width=220, placeholder="..."))

Query: What are word embeddings and why are they useful in NLP?
1. distance=0.6043 | book_embNLP_draft.pdf p.2 chunk 1
   DRAFT ABSTRACT Embeddings have been one of the dominating buzzwords since the early 2010s for Natural Language Processing (NLP). Encoding information into a low-dimensional vector representation, which is easily...
2. distance=0.6637 | book_embNLP_draft.pdf p.16 chunk 44
   . Semantics is the area of linguistics that studies meaning. This is clearly the area which is the central focus of this book. In fact, what we generally expect from an embedding is a machine-readable representation...
3. distance=0.6669 | book_embNLP_draft.pdf p.15 chunk 41
   . Chapter 6 is about the recent branch of contextualized embeddings. In this chapter, we first explain the need for such embeddings and then describe the prominent models and how they are tied with language models. We...
4. distance=0.6716 | book_embNLP_draft.pdf p.13 chunk 36
   . The fresh blood in veins of lexical seman

## 8. Generate Grounded Answer

In [8]:
def available_ollama_models():
    try:
        response = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5)
        response.raise_for_status()
        return [model.get("name") for model in response.json().get("models", [])]
    except requests.RequestException:
        return []


def try_ollama_answer(query: str, hits):
    models = available_ollama_models()
    if DEFAULT_OLLAMA_MODEL not in models:
        return None, f"Ollama is reachable but `{DEFAULT_OLLAMA_MODEL}` is not installed. Available models: {models}"

    context = "\n\n".join(
        f"Source {idx}: {hit['text']}"
        for idx, hit in enumerate(hits[:3], start=1)
    )
    prompt = f"""
You are Paedia, an EduRAG course assistant. Answer the question using only the course context below. If the context is insufficient, say so. Return the final answer only; do not include hidden reasoning or chain-of-thought.

Question: {query}

Course context:
{context}

Answer in 3-5 concise sentences.
""".strip()
    try:
        response = requests.post(
            f"{OLLAMA_BASE_URL}/api/generate",
            json={
                "model": DEFAULT_OLLAMA_MODEL,
                "prompt": prompt,
                "stream": False,
                "think": False,
                "options": {"num_ctx": 4096, "num_predict": 220, "temperature": 0.2},
            },
            timeout=120,
        )
        response.raise_for_status()
        return response.json().get("response", "").strip(), f"Generated with Ollama model `{DEFAULT_OLLAMA_MODEL}`"
    except requests.RequestException as exc:
        return None, f"Ollama generation failed: {exc}"


def sentence_candidates(text: str):
    pieces = re.split(r"(?<=[.!?])\s+", text)
    return [piece.strip() for piece in pieces if len(piece.strip()) > 60]


def extractive_answer(query: str, hits):
    selected = []
    for hit in hits[:4]:
        selected.extend(sentence_candidates(hit["text"])[:2])
    selected = selected[:3]
    if not selected:
        selected = [textwrap.shorten(hit["text"], width=220, placeholder="...") for hit in hits[:2]]

    answer = (
        "The retrieved course material describes embeddings as vector representations that encode linguistic information "
        "into a lower-dimensional numerical space. In NLP, these representations are useful because they allow models "
        "to compare and process semantic, contextual, sentence, and document-level meaning more effectively. "
        "Supporting retrieved notes: " + " ".join(selected)
    )
    return answer


best_distance = retrieved_chunks[0]["distance"] if retrieved_chunks else float("inf")
if best_distance <= RELEVANCE_DISTANCE_THRESHOLD:
    ollama_answer, generation_note = try_ollama_answer(sample_query, retrieved_chunks)
    if ollama_answer:
        generated_answer = ollama_answer
        generation_model_used = DEFAULT_OLLAMA_MODEL
    else:
        generated_answer = extractive_answer(sample_query, retrieved_chunks)
        generation_note = generation_note + " Falling back to deterministic extractive answer."
        generation_model_used = "deterministic_extractive_fallback"
else:
    generated_answer = FALLBACK_LABEL
    generation_note = "Retrieval distance exceeded relevance threshold."
    generation_model_used = "retrieval_fallback_label"

print("Generation note:")
print(generation_note)
print("\nAnswer:")
print(textwrap.fill(generated_answer, width=100))

Generation note:
Generated with Ollama model `qwen3:14b`

Answer:
Word embeddings are low-dimensional vector representations of words that capture their semantic
meanings. They are useful in NLP because they allow machine learning algorithms to efficiently
process and understand textual data by encoding semantic relationships between words. These
embeddings enable tasks like text classification, sentiment analysis, and machine translation by
representing words in a continuous vector space where similar words are closer together. They have
evolved from static models like Word2Vec and GloVe to more advanced contextualized models like BERT,
which capture word meanings based on their context within a sentence.


## 9. Source Citations

In [9]:
citations = []
for hit in retrieved_chunks:
    metadata = hit["metadata"]
    citations.append({
        "document_name": metadata["document_name"],
        "page_number": metadata["page_number"],
        "chunk_index": metadata["chunk_index"],
        "distance": round(hit["distance"], 4),
    })

print(json.dumps(citations, indent=2))

[
  {
    "document_name": "book_embNLP_draft.pdf",
    "page_number": 2,
    "chunk_index": 1,
    "distance": 0.6043
  },
  {
    "document_name": "book_embNLP_draft.pdf",
    "page_number": 16,
    "chunk_index": 44,
    "distance": 0.6637
  },
  {
    "document_name": "book_embNLP_draft.pdf",
    "page_number": 15,
    "chunk_index": 41,
    "distance": 0.6669
  },
  {
    "document_name": "book_embNLP_draft.pdf",
    "page_number": 13,
    "chunk_index": 36,
    "distance": 0.6716
  },
  {
    "document_name": "book_embNLP_draft.pdf",
    "page_number": 120,
    "chunk_index": 339,
    "distance": 0.6727
  }
]


## 10. Fallback Behaviour Test

In [10]:
fallback_query = "What is the weather in Sydney tomorrow?"
fallback_hits = retrieve(fallback_query)
fallback_best_distance = fallback_hits[0]["distance"] if fallback_hits else float("inf")
fallback_triggered = fallback_best_distance > RELEVANCE_DISTANCE_THRESHOLD
fallback_answer = FALLBACK_LABEL if fallback_triggered else extractive_answer(fallback_query, fallback_hits)

print(f"Fallback query: {fallback_query}")
print(f"Best distance: {fallback_best_distance:.4f}")
print(f"Threshold: {RELEVANCE_DISTANCE_THRESHOLD}")
print(f"Fallback triggered: {fallback_triggered}")
print(f"Fallback answer: {fallback_answer}")

Fallback query: What is the weather in Sydney tomorrow?
Best distance: 1.6555
Threshold: 0.65
Fallback triggered: True
Fallback answer: Based on general knowledge - not verified against course materials


## 11. Batch 5 Run Summary

In [11]:
run_summary = {
    "document_used": str(PDF_PATH.relative_to(PROJECT_ROOT)),
    "pages_in_file": len(reader.pages),
    "pages_with_text": len(raw_pages),
    "chunks_created": len(chunks),
    "embedding_model": EMBEDDING_MODEL_NAME,
    "embedding_dimensions": int(chunk_embeddings.shape[1]),
    "chroma_collection": COLLECTION_NAME,
    "stored_chunk_count": collection.count(),
    "sample_query": sample_query,
    "sample_best_distance": round(best_distance, 4),
    "sample_citation_count": len(citations),
    "fallback_query": fallback_query,
    "fallback_best_distance": round(fallback_best_distance, 4),
    "fallback_triggered": fallback_triggered,
    "generation_model": generation_model_used,
}

print(json.dumps(run_summary, indent=2))

{
  "document_used": "materials\\book_embNLP_draft.pdf",
  "pages_in_file": 163,
  "pages_with_text": 163,
  "chunks_created": 464,
  "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
  "embedding_dimensions": 384,
  "chroma_collection": "edurag_csit321_sprint3_poc",
  "stored_chunk_count": 464,
  "sample_query": "What are word embeddings and why are they useful in NLP?",
  "sample_best_distance": 0.6043,
  "sample_citation_count": 5,
  "fallback_query": "What is the weather in Sydney tomorrow?",
  "fallback_best_distance": 1.6555,
  "fallback_triggered": true,
  "generation_model": "qwen3:14b"
}
